# 03 — Two-Stage Retrieval: Cross-Encoder Reranking

**Track:** Intermediate · **Stage:** Retrieval Engineering

First-stage retrieval (Dense/Sparse/Hybrid) optimizes for recall and speed over millions of documents. Reranking optimizes for precision over a small candidate set (e.g., top 20). 

In this deep dive, you will build a two-stage pipeline. A fast vector search will retrieve candidates, and a heavier **Cross-Encoder** will re-score them based on deep semantic interaction between the query and the document.

## Setup: LangChain Rerankers

We will use `CrossEncoderReranker` from LangChain, wrapping a local HuggingFace cross-encoder model.

In [ ]:
# !pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

def print_results(results):
    for i, doc in enumerate(results):
        score_text = f" | Relevance Score: {doc.metadata.get('relevance_score', 'N/A'):.4f}" if 'relevance_score' in doc.metadata else ""
        print(f"[{i+1}] Source: {doc.metadata['source']}{score_text}")
        print(f"{doc.page_content}\n")

## 1. The Hard Corpus

We need a corpus designed to trick a standard dense or sparse retriever. 
We will ask: *"Does the Atlas supplier need to comply with Regulation R-17?"*

- `doc1` mentions Atlas and suppliers, but not R-17.
- `doc2` mentions R-17 and Atlas, but not suppliers.
- `doc3` is the exact answer, but uses different phrasing.

In [ ]:
corpus = [
    Document(
        page_content="Atlas Project Onboarding: We require all new suppliers for the Atlas project to sign standard NDAs.",
        metadata={"source": "atlas_onboarding.md"}
    ),
    Document(
        page_content="Compliance R-17 Overview: Regulation R-17 applies to all internal databases. The Atlas internal nodes are fully compliant.",
        metadata={"source": "compliance_r17.md"}
    ),
    Document(
        page_content="Vendor Requirements: Any third-party company providing database infrastructure (like DataStax) must undergo an R-17 audit.",
        metadata={"source": "vendor_requirements.md"}
    )
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(corpus, embeddings)

# First stage: Retrieve top 3
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## 2. First-Stage Retrieval Failure

Let's see how standard dense retrieval ranks the documents. 
Notice that the most relevant document (`vendor_requirements.md`) might rank last because it lacks the word "Atlas" and "supplier", even though "third-party company providing database infrastructure" is semantically the answer.

In [ ]:
question = "Does the Atlas supplier need to comply with Regulation R-17?"

print("--- Stage 1: Base Retrieval (No Reranking) ---")
base_results = base_retriever.invoke(question)
print_results(base_results)

## 3. Second-Stage Cross-Encoder Reranking

Unlike bi-encoders (which embed the query and document separately), a cross-encoder passes the query and document *together* through the transformer network. It is much slower, which is why we only run it on the top `k` candidates from stage 1.

In [ ]:
# Initialize a small local cross-encoder model
model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
compressor = CrossEncoderReranker(model=model, top_n=2)

# Wrap the base retriever with the reranker
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

print("--- Stage 2: Reranked Results ---")
reranked_results = compression_retriever.invoke(question)
print_results(reranked_results)

## Reflection

1. **Relevance Scores:** Notice the `relevance_score` attached to the reranked metadata. Cross-encoders output a logit score representing true relevance. If all top candidates have terrible scores, you can use this as an early abstention trigger before calling an LLM.
2. **Latency vs Quality:** A cross-encoder adds significant latency (often 200-500ms). Never run it across your whole database. Always pair it with a fast first-stage retriever.